# 🏠 Problema Prático — Previsão de Preço de Imóveis com ANN

## Contexto

O dataset **California Housing** contém dados de 20.640 blocos residenciais da Califórnia (censo de 1990).  
O objetivo é prever o **valor mediano dos imóveis** (em US$ 100 mil) a partir de características do bairro.

Este é um problema de **regressão**: a saída é um número contínuo.

---

## Variáveis do dataset

| Variável | Descrição |
|---|---|
| `MedInc` | Renda mediana dos moradores |
| `HouseAge` | Idade mediana das casas (anos) |
| `AveRooms` | Média de cômodos por residência |
| `AveBedrms` | Média de quartos por residência |
| `Population` | População do bloco |
| `AveOccup` | Média de moradores por residência |
| `Latitude` | Latitude geográfica |
| `Longitude` | Longitude geográfica |
| **`MedHouseVal`** | **Valor mediano dos imóveis — variável alvo** |

---

## Fluxo do problema

```
[MedInc, HouseAge, AveRooms, ...]  →  [ ANN ]  →  preço previsto (US$ 100k)
          8 variáveis de entrada                       1 saída contínua
```

---
## Passo 1 — Carregar e explorar os dados

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_california_housing

# Carregar dataset (já vem no Scikit-learn, sem download externo)
dados = fetch_california_housing(as_frame=True)
df = dados.frame

print(f"Tamanho do dataset: {df.shape[0]} amostras, {df.shape[1]} colunas")
print()
df.head()

In [ ]:
# Estatísticas básicas
df.describe().round(2)

In [ ]:
# Distribuição da variável alvo
plt.figure(figsize=(8, 4))
plt.hist(df['MedHouseVal'], bins=50, color='#185FA5', edgecolor='white', linewidth=0.5)
plt.xlabel('Valor mediano dos imóveis (US$ 100k)')
plt.ylabel('Frequência')
plt.title('Distribuição dos preços de imóveis — California Housing')
plt.tight_layout()
plt.show()

print(f"Preço mínimo : US$ {df['MedHouseVal'].min()*100:.0f}k")
print(f"Preço máximo : US$ {df['MedHouseVal'].max()*100:.0f}k")
print(f"Preço médio  : US$ {df['MedHouseVal'].mean()*100:.0f}k")

---
## Passo 2 — Preparar os dados

Antes de treinar a ANN, precisamos:

1. **Separar entradas (X) e saída (y)**
2. **Dividir em treino e teste** — o modelo aprende com o treino e é avaliado no teste (dados nunca vistos)
3. **Normalizar as entradas** — redes neurais são sensíveis à escala. Com `StandardScaler`, cada variável fica com média 0 e desvio padrão 1

$$x_{norm} = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separar entradas e saída
X = df.drop(columns='MedHouseVal').values
y = df['MedHouseVal'].values

# Dividir treino (80%) e teste (20%)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalizar as entradas
scaler = StandardScaler()
X_treino = scaler.fit_transform(X_treino)   # aprende média e desvio no treino
X_teste  = scaler.transform(X_teste)        # aplica a mesma escala no teste

print(f"Treino : {X_treino.shape[0]} amostras")
print(f"Teste  : {X_teste.shape[0]} amostras")
print(f"Entradas: {X_treino.shape[1]} variáveis")

---
## Passo 3 — Treinar a Rede Neural

Usamos o `MLPRegressor` do Scikit-learn.  
A arquitetura escolhida é: **8 entradas → 64 → 32 → 1 saída**

```
Entrada (8)  →  Camada oculta 1 (64 neurônios)  →  Camada oculta 2 (32 neurônios)  →  Saída (1)
```

Parâmetros principais:

| Parâmetro | Valor | Significado |
|---|---|---|
| `hidden_layer_sizes` | `(64, 32)` | Arquitetura das camadas ocultas |
| `activation` | `'relu'` | Função de ativação nas camadas ocultas |
| `max_iter` | `500` | Número máximo de épocas |
| `learning_rate_init` | `0.001` | Taxa de aprendizado inicial |

In [ ]:
from sklearn.neural_network import MLPRegressor

ann = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    max_iter=500,
    learning_rate_init=0.001,
    random_state=42,
    verbose=False
)

ann.fit(X_treino, y_treino)
print("Treinamento concluído!")
print(f"Épocas realizadas: {ann.n_iter_}")

---
## Passo 4 — Avaliar o modelo

Métricas usadas:

| Métrica | Fórmula | Interpretação |
|---|---|---|
| **R²** | $1 - \frac{\sum(y-\hat{y})^2}{\sum(y-\bar{y})^2}$ | 1.0 = perfeito, 0 = inútil |
| **RMSE** | $\sqrt{\frac{1}{N}\sum(y-\hat{y})^2}$ | Erro médio na mesma unidade da saída |
| **MAE** | $\frac{1}{N}\sum|y-\hat{y}|$ | Erro absoluto médio |

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred_treino = ann.predict(X_treino)
y_pred_teste  = ann.predict(X_teste)

r2_tr  = r2_score(y_treino, y_pred_treino)
r2_te  = r2_score(y_teste,  y_pred_teste)
rmse   = np.sqrt(mean_squared_error(y_teste, y_pred_teste))
mae    = mean_absolute_error(y_teste, y_pred_teste)

print("========== Resultados ==========")
print(f"R²   treino : {r2_tr:.4f}")
print(f"R²   teste  : {r2_te:.4f}")
print(f"RMSE teste  : {rmse:.4f}  (× US$100k = ±US${rmse*100:.0f}k de erro médio)")
print(f"MAE  teste  : {mae:.4f}  (× US$100k = ±US${mae*100:.0f}k de erro médio)")

In [ ]:
# Gráfico: valor real vs valor previsto
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Scatter: real vs previsto ---
ax = axes[0]
ax.scatter(y_teste, y_pred_teste, alpha=0.3, s=10, color='#185FA5')
lims = [0, 5.5]
ax.plot(lims, lims, 'r--', lw=1.5, label='Previsão perfeita')
ax.set_xlabel('Valor real (US$ 100k)')
ax.set_ylabel('Valor previsto (US$ 100k)')
ax.set_title(f'Real vs Previsto — R² = {r2_te:.3f}')
ax.legend()

# --- Distribuição dos erros ---
ax = axes[1]
erros = y_teste - y_pred_teste
ax.hist(erros, bins=60, color='#993C1D', edgecolor='white', linewidth=0.4)
ax.axvline(0, color='k', ls='--', lw=1.5)
ax.set_xlabel('Erro (real − previsto) em US$ 100k')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição dos erros de previsão')

plt.suptitle('Avaliação do Modelo — ANN Regressão', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Passo 5 — Curva de aprendizado

A **curva de loss** mostra como o erro foi diminuindo ao longo das épocas de treinamento.  
Se a curva estabilizar, significa que a rede convergiu.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ann.loss_curve_, color='#185FA5', lw=2)
plt.xlabel('Época')
plt.ylabel('Loss (MSE)')
plt.title('Curva de aprendizado da ANN')
plt.yscale('log')
plt.tight_layout()
plt.show()

print(f"Loss inicial : {ann.loss_curve_[0]:.4f}")
print(f"Loss final   : {ann.loss_curve_[-1]:.4f}")
print(f"Redução      : {(1 - ann.loss_curve_[-1]/ann.loss_curve_[0])*100:.1f}%")

---
## 🧪 Exercícios

Experimente modificar o código e observe o impacto nas métricas:

**1. Altere a arquitetura da rede:**
```python
hidden_layer_sizes=(16,)          # rede menor — underfitting?
hidden_layer_sizes=(128, 64, 32)  # rede maior — overfitting?
```

**2. Troque a função de ativação:**
```python
activation='tanh'    # compara com relu
activation='logistic'
```

**3. Retire a normalização** (comente o `StandardScaler`) e veja o que acontece com o R².

**4. Reduza os dados de treino** para 10% e compare o R² com 80%:
```python
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.9, random_state=42)
```

---

## 💡 Discussão

- Por que é importante avaliar no **conjunto de teste** e não no de treino?
- O que indica um R² alto no treino e baixo no teste? (overfitting)
- Por que a **normalização** é fundamental para redes neurais?
- Esse modelo poderia ser usado em um sistema de controle? Em qual contexto?